# Cleaning for eHRAF Scraper
The current stats are merely to 
- Clean and reorganize the dataframe 
- Get datasets that correspond to OCM pairs (optional)
- Find the most common OCM codes
- Find association rules (when one OCM appears this other OCM is likely to appear) 


## Clean the Dataframe


In [1]:
import pandas as pd                 # dataframe storing
import numpy as np
import re                           # regex for searching through strings
# import mlxtend                      # for coocurrance exploration
import copy
import os

# strip OCMs so they become a list again
def OCM_stripper(df, OCM='OCM'):
    if type(df[OCM].iloc[0]) is list: # if already a list, return without alteration
        return df
    df_ocm = df.copy() # so that the original dataframe is not affected
    df_ocm[OCM] = df_ocm[OCM].apply(lambda x: re.sub(" |\'",'',x))
    df_ocm[OCM] = df_ocm[OCM].apply(lambda x: x[1:-1].split(','))
    return df_ocm


### Load data (<font color="red">Only run one cell</font>)

#### Load Single Scraping (Optional)


In [2]:

# CHANGE: put folder name here (use the multi-dataset code if you have multiple scrapings you wish to combine)–
folder = r'subjects-(sickness)_FILTERS-culture_level_samples(PSF)'                          # OCM 750
# folder = r'subjects-(religious_practices_OR_sickness)_FILTERS-culture_level_samples(PSF)'   # OCM 750 + 780
# CHANGE: edit folder location as necessary-
directory = '../Data/' + folder


df_raw = pd.read_excel(directory + '/_Altogether_Dataset.xlsx')
redonePassageNums_bool = False # Here for later code which adds a second column to the dataset_lists file if needed

df_raw = OCM_stripper(df_raw)
# set up read me text for later
readme_text = 'The following post processing files (that is all folder and files excluding those produced from the scraper such as raw Culture files and _Altogether_Dataset.xlsx) were done on a single dataset/scraping\n'

print("Num Passages - ", len(df_raw))
# did it work? did it output a single OCM string?
df_raw['OCM'][0][0]


Num Passages -  42555


'752'

#### Load Multiple Scraping (Optional)

In [3]:
# CHANGE: put the folders of the data you wish to combine, Do not run this cell if you plan on using the cell above–

folder1 = r'subjects-(religious_practices_OR_sickness)_FILTERS-culture_level_samples(PSF)'   # OCM ['750' , '780']
folder2 = r'(subjects-(contracts_OR_disabilities_OR_disasters_OR_friendships_OR_gift_giving_OR_infant_feeding_OR_lineages_OR_etc'   # OCM ['586' , '684' , '688' , '731' , '732' , '756' , '767' , '777' , '791' , '792' , '793' , '431' , '572' , '594' , '613' , '624' , '675' , '853']
folder3 = r''
folder4 = r''   
folder_loop = [folder1, folder2, folder3, folder4]
# CHANGE: edit folder location as necessary-
directory = '../Data/'


print("Num Passages:")
readme_text = 'The following post processing files (that is all folder and files excluding those produced from the scraper such as raw Culture files and _Altogether_Dataset.xlsx) were done on a COMBINED dataset of multiple scrapings. Here are the folders used:\n'
df_raw = pd.DataFrame()
for index, folder in enumerate(folder_loop):
    if len(folder) ==0:
        continue
    file_loc = directory + folder + '/_Altogether_Dataset.xlsx'
    df_raw_input = pd.read_excel(file_loc)
    df_raw = pd.concat([df_raw, df_raw_input], ignore_index=True)
    latest_dir = directory + folder # for use of dataset later
    readme_text += f'\tfolder {index+1}: {folder}\n'
    print(f"Folder {index+1} - ", len(df_raw_input))
directory = latest_dir #make the last directory the directory that all the new files will go

# Redo Passage Numbers
redonePassageNums_bool = True # Here for later code which adds a second column to the dataset_lists file if needed
origPassageNums = df_raw['Passage Number']
df_raw['Passage Number'] = df_raw.index+1 # since there may be overlap in IDs, create new and unique passage IDs for each passage
df_raw = OCM_stripper(df_raw)


print("Total - ", len(df_raw))

Num Passages:
Folder 1 -  82202
Folder 2 -  50493
Total -  132695


### Remove 'run_info' column

In [33]:
df_raw = df_raw.drop(['run_Info'], axis=1)

### Remove blank passages

In [34]:
# drop all rows that have a blank passage
print(f'Before: {len(df_raw)}')
df_raw = df_raw.dropna(subset="Passage")
print(f'After: {len(df_raw)}')

Before: 132695
After: 132316


### Remove Duplicates

Currently, duplicate passages will be removed (keep only one) regardless of if they have different or same OCMs <br>
Previously, only passages will be removed if they contain a duplicate passage with the same OCMs. Meaning duplicate passages with different OCMs would have remained

In [35]:
# (exploratory)  
df_dummy = copy.deepcopy(df_raw)
# Find all passages which are duplicates but do not share the same document. 
# First let's explore some of the duplicates
dup1 = df_dummy["Passage"].duplicated(keep=False)  # find all duplicate passages
dup2 = df_dummy[dup1].duplicated(subset=["Passage", "DocTitle"], keep=False) #of the duplicate passages, find those that shair a passage and doc title
# rows which contain duplicate passages but not part of the same document (only top 4 shown)
print(f'Number of passages whose duplicates come from different documents: {len(df_dummy[dup1][~dup2].sort_values(by="Passage"))}')
df_dummy[dup1][~dup2].sort_values(by='Passage').head(4) #Note that there may be 3 or more instances of a particular duplicate passage but not all may have the same doc so this line will show what seems to be a single passage that does not have a duplicate.


Number of passages whose duplicates come from different documents: 100


,Passage Number,Region,SubRegion,Culture,DocTitle,Section,Author,Page,Year,OCM,OWC,Passage
113011,113012,Africa,Western Africa,Wolof,Senegal in former times: second study on Cayor...,The Diop and The Dieng,"Rousseau, R.",38,1941,"[613, 631]",ms30,/135/
110951,110952,Oceania,Micronesia,Chuuk,"The inhabitants of the Truk Islands: religion,...",6.,"Bollig, Laurentius",148,1927,"[484, 539, 791, 825]",or19,/135/
92880,92881,Asia,Southeast Asia,Ifugao,The Mayawyaw ritual: VII. hunting and its ritual,The To’b-ag- Rites,"Lambrecht, Francis",20,1957,"[224, 539, 773, 778, 793]",oa19,"1. Are living Bugan and Wigan at Dukligan,/ we..."
92879,92880,Asia,Southeast Asia,Ifugao,The Mayawyaw ritual: VII. hunting and its ritual,The To’b-ag- Rites,"Lambrecht, Francis",20,1957,"[224, 539, 773, 778, 793]",oa19,"1. Mata’gu cha Bu’gan ya Wi’gan ad Chu-li’gan,..."


In [36]:
# (exploratory) 
# Find passages which have duplicates but whose duplicates come from different cultures
df_dummy["OCM"] = df_dummy['OCM'].apply(tuple) #turn the OCM list to a tuple to allow for comparisons

# Of the passages which have duplicates, find and keep all which have the same OCM
dup3 = df_dummy[dup1].duplicated(subset=["Passage", "OCM"], keep=False)
# Show only the passages with duplicates but NOT matching OCMs
print(f'Number of passages whose duplicates do not share OCMs:  {len(df_dummy[dup1][~dup3].sort_values(by="Passage"))}')
df_dummy["OCM"] = df_dummy['OCM'].apply(list)
df_dummy[dup1][~dup3].sort_values(by="Passage").head(4) #same quirk as above

Number of passages whose duplicates do not share OCMs:  562


,Passage Number,Region,SubRegion,Culture,DocTitle,Section,Author,Page,Year,OCM,OWC,Passage
39013,39014,North-America,Eastern Woodlands,Iroquois,"The code of Handsome Lake, the Seneca prophet",SECTION 45,"Parker, Arthur C.",45,1913,"[539, 776, 779, 783]",nm09,"""`Now another message for your people."
39002,39003,North-America,Eastern Woodlands,Iroquois,"The code of Handsome Lake, the Seneca prophet",SECTION 38,"Parker, Arthur C.",43,1913,"[539, 775, 779, 783]",nm09,"""`Now another message for your people."
38988,38989,North-America,Eastern Woodlands,Iroquois,"The code of Handsome Lake, the Seneca prophet",SECTION 35,"Parker, Arthur C.",42,1913,"[539, 776, 779, 783, 796]",nm09,"""`Now another message to tell your people."
38992,38993,North-America,Eastern Woodlands,Iroquois,"The code of Handsome Lake, the Seneca prophet",SECTION 36,"Parker, Arthur C.",42,1913,"[539, 779, 788, 793]",nm09,"""`Now another message to tell your people."


In [37]:
# (exploratory, but important to check!) 
# Find passages which have duplicates but whose duplicates have different OCM numbers
df_dummy["OCM"] = df_dummy['OCM'].apply(tuple) #turn the OCM list to a tuple to allow for comparisons

# Find all passages which are duplicates but do not share the same culture. 
# First let's explore some of the duplicates
dup1 = df_dummy["Passage"].duplicated(keep=False)  # find all duplicate passages
dup2 = df_dummy[dup1].duplicated(subset=["Passage", "Culture"], keep=False) #of the duplicate passages, find those that shair a passage and doc title
# rows which contain duplicate passages but not part of the same culture
print(f'Number of passages whose duplicates come from different Cultures: {len(df_dummy[dup1][~dup2].sort_values(by="Passage"))}')
df_dummy[dup1][~dup2].sort_values(by='Passage') # NOTE Make sure these are all ones you are okay with deleting because there could be circumstances where a passage is shaired between cultures but is ultimately relevant.

Number of passages whose duplicates come from different Cultures: 11


,Passage Number,Region,SubRegion,Culture,DocTitle,Section,Author,Page,Year,OCM,OWC,Passage
110951,110952,Oceania,Micronesia,Chuuk,"The inhabitants of the Truk Islands: religion,...",6.,"Bollig, Laurentius",148,1927,"(484, 539, 791, 825)",or19,/135/
113011,113012,Africa,Western Africa,Wolof,Senegal in former times: second study on Cayor...,The Diop and The Dieng,"Rousseau, R.",38,1941,"(613, 631)",ms30,/135/
34979,34980,Asia,Southeast Asia,Iban,Iban shamanism: an analysis of the ethnographi...,II,"Graham, Penelope",132,1987,"(756, 775, 776, 787)",oc06,"Again,"
122443,122444,Oceania,Melanesia,Trobriands,Politics of the kula ring: an analysis of the ...,Alternative to Malinowski's Interpretation,"Uberoi, J. P. Singh",39,1971,"(554, 613, 773)",ol06,"Again,"
1865,1866,South-America,Southern South America,Mataco,Ceremonies for the expulsion of illnesses amon...,Untitled Section,"Dijour, élisabeth",3,1933,"(755, 756, 783, 796)",si07,Notes
6967,6968,Africa,Eastern Africa,Somali,The terminology and practice of Somali weather...,9.77 District VII: WAJEER (The N. F. D.),"Galaal, Muusa H. I.",68,1968,"(184, 787, 805, 821)",mo04,Notes
3631,3632,South-America,Northwestern South America,Kogi,The sacred mountain of Colombia's Kogi Indians,CATALOGUE OF ILLUSTRATIONS,"Reichel-Dolmatoff, Gerardo",28,1990,"(342, 361, 778, 787)",sc07,Plate XXIV
3635,3636,South-America,Northwestern South America,Kogi,The sacred mountain of Colombia's Kogi Indians,CATALOGUE OF ILLUSTRATIONS,"Reichel-Dolmatoff, Gerardo",30,1990,"(291, 516, 788)",sc07,Plate XXX
3637,3638,South-America,Northwestern South America,Kogi,The sacred mountain of Colombia's Kogi Indians,CATALOGUE OF ILLUSTRATIONS,"Reichel-Dolmatoff, Gerardo",30,1990,"(211, 291, 533, 782)",sc07,Plate XXXI
18499,18500,Oceania,Polynesia,Tikopia,Chapter 12,Possession.,"Rivers, W. H. R.",321,1914,"(787,)",ot11,“Yes.”


In [38]:
# remove all duplicated passages
# df_raw["OCM"] = df_raw['OCM'].apply(tuple) #turn the OCM list to a tuple to allow for comparisons (this is here when we used to keep duplicated passages with different OCMs)

# drop duplicates
print(f'Before {len(df_raw)}')
df_raw.drop_duplicates(subset=["Passage"], keep='first', inplace=True)
print(f'After {len(df_raw)}')

# df_raw["OCM"] = df_raw['OCM'].apply(list) #turn the OCM back to a list (this is here when we used to keep duplicated passages with different OCMs)

Before 132316
After 111730


### Remove passages that are too long 

THIS IS RECOMMENDED FOR MACHINE LEARNING SO IF YOU DON'T CARE ABOUT THE PASSAGE BEING TOO LONG AND WANT LONG PASSAGES CODED, DO NOT RUN THIS CELL

In [39]:
# CHANGE the cut off at will, the max token limit for BERT is 512 but assume that some passages will have extra tokens because of weird characters or long words (etiology become et## and ##iology tokens).
cut_off = 425 

mask = df_raw['Passage'].apply(lambda x: len(x.split())<=cut_off)
print(f"Percentage of too long passages {round(100*(1 - sum(mask)/(mask.count())),1)}%")
print(f'Before {len(df_raw)}')
df_raw = df_raw.loc[mask]
print(f'After {len(df_raw)}')


Percentage of too long passages 2.4%
Before 111730
After 109022


### Remove extra OCMs

In [40]:
# make the OCM into a valid format
def OCM_validity_checker(OCM_list):
    assert isinstance(OCM_list, list), "Need to insert a list"
    OCM_list = [str(OCM) for OCM in OCM_list]
    return OCM_list

# cut down and use only the OCMs we want
def OCM_remover(df, OCM_list:list, OCM_Dataset_A=False, OCM_Dataset_B=False, saveComparison=False):
    df_main = df

    # change to a list of strings if not already
    OCM_list = OCM_validity_checker(OCM_list)

    # If you use a higher order code (750) eHRAF attempts to aquire ALL OCMs related to your input.
    # select only the OCMs we originally wished to search for by inputting OCM's into a list
    msk = df_main['OCM'].apply(lambda x: not set(x).isdisjoint(OCM_list))
    df_main = df_main.loc[msk]

    print(f"Total passages after reducing\n{len(df_main)}\n")
    # If you want to compare this list with overlap (like if you have basically two searches stuck together) 
    if OCM_Dataset_A is not False and OCM_Dataset_B is not False:
        df_A = df
        df_B = df
        OCM_Dataset_A = OCM_validity_checker(OCM_Dataset_A)
        OCM_Dataset_B = OCM_validity_checker(OCM_Dataset_B)
        # get counts for both datasets
        msk = df_A['OCM'].apply(lambda x: not set(x).isdisjoint(OCM_Dataset_A))
        df_A = df_A.loc[msk]
        msk = df_B['OCM'].apply(lambda x: not set(x).isdisjoint(OCM_Dataset_B))
        df_B = df_B.loc[msk]

        msk = df_A['OCM'].apply(lambda x: not set(x).isdisjoint(OCM_Dataset_B))
        df_dummy_AB = df_A.loc[msk]

        # Go the extra step and save a comparison dataframe to excel showing the how one list's OCM's compare to the other's 
        if saveComparison is True:
            value_counts_A = df_A["Culture"].value_counts(ascending=True)
            value_counts_B = df_B["Culture"].value_counts(ascending=True)

            culture_order = value_counts_A.index

            values_A = value_counts_A.values
            values_B = value_counts_B.reindex(culture_order, fill_value=0).values #reindex so they match
            # get first value
            firstVal_A = OCM_Dataset_A[0]
            firstVal_B = OCM_Dataset_B[0]

            df_valcon =  pd.DataFrame({firstVal_A: values_A, firstVal_B: values_B}, index=culture_order)
            df_valcon["percentage"] = round(df_valcon[firstVal_A] / (df_valcon[firstVal_A] + df_valcon[firstVal_B]),2)
            df_valcon["log abs ratio"] = round(np.abs(np.log(df_valcon[firstVal_A] / df_valcon[firstVal_B])),2)
            print(f'Passages of first Dataset with the OCMs {OCM_Dataset_A}:\n{len(df_A)}')
            print(f'Passages of second Dataset with the OCMs {OCM_Dataset_B}:\n{len(df_B)}')
            print(f'First dataset\'s overlap with second dataset:\n{len(df_dummy_AB)}\n\n')
            print("Number of cultures per dataset and comparison dataframe:")
            print(df_valcon)
            df_valcon.to_excel(directory +"/_Assignment_counts.xlsx")

    else:
        print(f'Passages after reducing OCMs only to the desired number:\n{len(df_main)}')
    return df_main


Read the comments carefully

In [ ]:

# CHANGE OCMs For your main filtering (MAKE SURE YOU ARE NOT RUNNING THIS UNLESS YOU AGREE WITH THE INPUTS)
# NOTE make sure you are including all OCMs you want to use in subsequent coding!!! 
# OCM_list = ["750", "751", "752", "753", "780", "781", "784", "785", '586' , '684' , '688' , '731' , '732' , '756' , '767' , '777' , '791' , '792' , '793' , '431' , '572' , '594' , '613' , '624' , '675' , '853'] 
OCM_list = ["750", "751", "752", "753", '784' , '731' , '732' , '777' , '791', '793']  #sicknes and non-sickness chosen OCMs
# OCM_list = ["750", "751", "752", "753"]  # Sickness OCMs

# (Optional) If your main dataset is comprised of multiple sub datasets, include them here merely for exploration and outputting a file which tells you the counts for each sub dataset
# ouptutes the file _Assignment_counts.xlsx
OCM_Dataset_A = ["750", "751", "752", "753"]                                                              # Sickness Dataset  
# OCM_Dataset_B = ["780", "781", "784", "785"]                                                              # 780 dataset
# OCM_Dataset_B = ['586' , '684' , '688' , '731' , '732' , '756' , '767' , '777' , '791' , '792' , '793']   # Theoretical Interest dataset
# OCM_Dataset_B = ['431' , '572' , '594' , '613' , '624' , '675' , '853']                                   # no-theoretical interest dataset
# OCM_Dataset_B = ['586' , '684' , '688' , '731' , '732' , '756' , '767' , '777' , '791' , '792' , '793' , '431' , '572' , '594' , '613' , '624' , '675' , '853'] # theoretical and non-theoretical
OCM_Dataset_B = ['784' , '731' , '732' , '777' , '791', '793']                                            # Non-Sickness dataset 


# UNCOMMENT ONE
# # Run for simple filtering (for single)
# df = OCM_remover(df_raw, OCM_list)
# Run for comparing and saving the difference between two datasets (for multi datasets)
df = OCM_remover(df_raw, OCM_list, OCM_Dataset_A, OCM_Dataset_B, saveComparison= True).copy() #copy() here to suppress the warning down below but is actually not necessary.


Total passages after reducing
22322

Passages of first Dataset with the OCMs ['750', '751', '752', '753']:
6112
Passages of second Dataset with the OCMs ['784', '731', '732', '777', '791', '793']:
17042
First dataset's overlap with second dataset:
832


Number of cultures per dataset and comparison dataframe:
                  750   784  percentage  log abs ratio
Culture                                               
Shluh               1    38        0.03           3.64
Highland Scots      7   114        0.06           2.79
Sinhalese          10    80        0.11           2.08
Kurds              10    73        0.12           1.99
Kanuri             14   180        0.07           2.55
Kapauku            17   109        0.13           1.86
Bemba              17   199        0.08           2.46
Bahia Brazilians   21   163        0.11           2.05
Guaraní            23    46        0.33           0.69
Wolof              23   382        0.06           2.81
Khasi              24    92  

### Shave OCMs

And make an exploded OCM dataframe

In [42]:


# Make a dataset in which each OCM have its own row by exploding (you can reset the index with .reset_index(drop=True))
df_OCM = df.explode(column='OCM').reset_index(drop=True)
# Find OCM's that do not fit the normal 100-900 OCM scheme
# NOTE 0 means the material is not relevant, I am unsure, however, why this sometimes appears with other OCM's in the same passage
# NOTE I believe 5310 and 5311 are different specifications of 531 while 1710 might be a more specific (and singlular) subset of 171? I do not believe the same for 77 and 1787
list_OCM = df_OCM['OCM'].value_counts().index.tolist()
small_OCM = [x for x in list_OCM if len(x) <3 or len(x) > 3]
print(f"OCMs too small or too large:\n{small_OCM}")


OCMs too small or too large:
['0', '5311', '5310', '1787', '72']


In [43]:
# CHANGE: add to the list for codes which should be removed (otherwise all others will be shaved)
remove_list = ['1787','7478', '77', '72']

# remove and shave OCM codes
print(f'starting list {len(df_OCM)}')
for i in remove_list:
    df_OCM = df_OCM[df_OCM["OCM"] != i]
# "Shave" the OCM codes that seem to have a parent (5310 and 5311 become 531).
df_OCM['OCM'] = df_OCM.OCM.apply(lambda x: x[0:3] if len(x) >= 3 else x)
print(f'Ending list {len(df_OCM)}')

starting list 87694
Ending list 87692


In [44]:
# Apply the removals like above to the original dataframe (this is easier than just imploding as there are duplicates which limit this)
print(f'Passages before shaving and duplicate removal:\n{len(df)}')
# remove specified OCM codes
df["OCM"] = df["OCM"].apply(lambda x: [item for item in x if item not in remove_list])
# shorten the 'small_OCM' OCMs so that 5310 becomes 531
df["OCM"] = df["OCM"].apply(lambda x: [item[0:3] if item in small_OCM else item for item in x])
print(f'Passages after shaving and duplicate removal:\n{len(df)}') #Note that this number should Probably not change from the above number
# explantaion of above list comprehension: go through every row of the column "OCM" (via apply) 
# lambda x is an anonymous function which takes the row "x" and inputs it into the function.
# each row has its list items iterated over ( "___ for item in x") and checked if each list item is part of the small_OCM list, if so,
# return the first 3 characters, if not, return the original list item. Return everything back as a list and apply it to the dataframe


list_OCM = df_OCM['OCM'].value_counts().index.tolist()
small_OCM = [x for x in list_OCM if len(x) <3 or len(x) > 3]
print(f"OCMs too small or too large:\n{small_OCM}")

Passages before shaving and duplicate removal:
22322
Passages after shaving and duplicate removal:
22322
OCMs too small or too large:
['0']


### Create Dictionary for later count comparisons

In [45]:
# Find the number of passages for each culture
culture_set = set(df["Culture"])
culture_dict = {}
it_count = 0
for cult_i in culture_set:
    row_count = len(df.loc[df["Culture"]==cult_i])
    culture_dict[cult_i] = row_count
    it_count += row_count
print(f'Passages in dictionary: \n{it_count}')


Passages in dictionary: 
22322


### Clean passage text

#### Remove [unknown] and [unavailable] from text

In [46]:
# Show the most common bracketed words (should see "[unknown]" or "[unavailable]" at the top)
def bracket_count(df=df, most_common=8):
    from collections import Counter
    bracket_set = []
    for pas in df["Passage"]:
        match = re.findall(r'\[.*?\]',pas)
        if len(match) >0:
            bracket_set += match

    counts = Counter(bracket_set)
    return counts.most_common(most_common) # N most common items
bracket_count(df,8)

[('[unknown]', 5746),
 ('[unavailable]', 708),
 ('[sic]', 20),
 ('[3]', 19),
 ('[1]', 18),
 ('[etc.]', 17),
 ('[2]', 16),
 ('[illegible]', 16)]

In [47]:
# Find an example  passage to use
pattern = r'\[unavailable\]'
filtered_df = df[df['Passage'].str.contains(pattern, case=False, regex=True)]
index_pas = filtered_df.index[1]

# Example of passage which needs cleaning (May be outdated depending on your original search query!)
print(f"Before:\n {df['Passage'][index_pas]}")
# Remove all "[unknown]", "[unavailable]", "[ a ]" or "[ i ]" text within the passages"
df["Passage"] = df["Passage"].apply(lambda x: re.sub(r"\[unknown\]|\[unavailable\]|\[ a \]|\[ i \]",'', x))
# after
print(f"After:\n {df['Passage'][index_pas]}")

Before:
 If in a village, during a certain time, several deaths have occurred, especially if an epidemic has been raging, the Mataco are in the habit of saying: [unknown]That place has been visited by many naút yil [unknown]. On the other hand, if in a certain village there has been no death and no disease for a long time, the Indians say: [unknown]In this place there are, at present, no nahút yil [unknown]. There is practically no disease and no illness which is not directly or indirectly ascribed by the Indians either to an aittáh slamsa or a nahút yil . Slight disorders and illnesses which are cured in a couple of days are ascribed to the aittáh, mortal diseases to the nahút . If a Mataco for instance breaks his leg, or if some other misfortune happens to him which does not end fatally, he is of opinion that he has been attacked by a [unknown]small[unknown] aittáh . Similarly, there are aittáh operating in stupendous natural phenomena, in hurricanes, in torrential rains, in thunder 

In [48]:
bracket_count(df,8)

[('[sic]', 20),
 ('[3]', 19),
 ('[1]', 18),
 ('[etc.]', 17),
 ('[2]', 16),
 ('[illegible]', 16),
 ('[JMR: evil spirits]', 13),
 ('[5]', 13)]

### Optional Exploration

In [137]:
# (OPTIONAL)
# Quick search for OCMs regardless of culture
# NOTE, sometimes a higher order code like 750 appears without lower order codes)
exclude_list = ['750','751','752','753', '784','731','732', '164','767'] #enter in the OCM strings you DON't want to see
include_list = ["624"] #enter your OCM strings of OCMs you want to see (passages only need to contain one, not all)


exc_msk = df['OCM'].apply(lambda x: set(x).isdisjoint(exclude_list))
exc = df.loc[exc_msk]
inc_msk = exc['OCM'].apply(lambda x: not set(x).isdisjoint(include_list))
inc = exc.loc[inc_msk]
inc

,Passage Number,Region,SubRegion,Culture,DocTitle,Section,Author,Page,Year,OCM,OWC,Passage
4842,4843,Africa,Eastern Africa,Maasai,An administrative survey of the Masai social s...,A. The Laigwanan and his Assistants -- choice ...,"Fosbrooke, H. A.",35,1948,"[561, 624, 789, 791]",fl12,The visits to the laibon of the embryo laigwan...
4870,4871,Africa,Eastern Africa,Maasai,Masai social customs,MASAI SOCIAL CUSTOMS,"Whitehouse, L. E.",149,1933,"[561, 624, 789, 791]",fl12,When the boys have elected their l'aigwenani t...
12018,12019,Africa,Western Africa,Hausa,Custom & politics in urban Africa: a study of ...,Battle of the Kola and Rebellion of the Friday...,"Cohen, Abner",136,1969,"[624, 771, 782, 788, 793]",ms12,According to the Maliki School of Islamic theo...
12078,12079,Africa,Western Africa,Hausa,Political support in a Hausa village,5.3 Notions about man.,"Faulkingham, Ralph Harold, 1943-",111,1971,"[623, 624, 626, 681, 754, 787, 791, 795]",ms12,"Up to 1960, a diviner who determined that his ..."
12181,12182,Africa,Western Africa,Hausa,Pilgrims in a strange land: Hausa communities ...,EARLY ISLAMIZATION OF CHAD,"Works, John A., 1944-",134,1976,"[184, 484, 624, 788, 793, 797]",ms12,Into this setting of a few scattered Islamic c...
...,...,...,...,...,...,...,...,...,...,...,...,...
124279,124280,North-America,Plains and Plateau,Pawnee,Ceremonies of the Pawnee,The Corn Planting Ceremony,"Murie, James R.",78,1989,"[624, 793, 796]",nq18,All this time the skull is in front of the alt...
124290,124291,North-America,Plains and Plateau,Pawnee,Ceremonies of the Pawnee,Young Corn Plant Ritual,"Murie, James R.",84,1989,"[243, 624, 793, 796]",nq18,"When the time has arrived, the priest of the l..."
124291,124292,North-America,Plains and Plateau,Pawnee,Ceremonies of the Pawnee,Young Corn Plant Ritual,"Murie, James R.",84,1989,"[243, 533, 624, 793, 796]",nq18,The priest now sends for this man; and when he...
124308,124309,North-America,Plains and Plateau,Pawnee,Ceremonies of the Pawnee,Young Corn Plant Ritual,"Murie, James R.",86,1989,"[243, 624, 793, 796]",nq18,"While the men were out after the corn plant, t..."


In [138]:
# Quick search for OCMs SUBINDEX BY ANOTHER COLUMN
include_list = ["159","451"] #enter your OCM strings of OCMs you want to see 
culture = "Akan" # enter the desired culture
msk = df.loc[df["Culture"]== culture]['OCM'].apply(lambda x: not set(x).isdisjoint(include_list))
out = df.loc[msk.index][msk]
out.head(4)

,Passage Number,Region,SubRegion,Culture,DocTitle,Section,Author,Page,Year,OCM,OWC,Passage
61050,61051,Africa,Western Africa,Akan,Search for security: an ethno-psychiatric stud...,The Making of an Obosomfo,"Field, M. J. (Margaret Joyce)",63,1970,"[159, 787, 793]",fe12,Of the practising obosom brafo priests who cam...
61051,61052,Africa,Western Africa,Akan,Search for security: an ethno-psychiatric stud...,The Making of an Obosomfo,"Field, M. J. (Margaret Joyce)",63,1970,"[159, 787, 793]",fe12,One said his obosom drove him into the wildern...
61052,61053,Africa,Western Africa,Akan,Search for security: an ethno-psychiatric stud...,The Making of an Obosomfo,"Field, M. J. (Margaret Joyce)",64,1970,"[159, 787, 793]",fe12,Another said he was five days in the bush befo...
61053,61054,Africa,Western Africa,Akan,Search for security: an ethno-psychiatric stud...,The Making of an Obosomfo,"Field, M. J. (Margaret Joyce)",64,1970,"[159, 776, 787, 793]",fe12,Another obosomfo —a woman with a son of about ...


In [139]:
# (OPTIONAL)
# There are some passages that describe previous passages but do not contain information themselves like: 
# "Notes" or "End" or "Log"
# This code cell indicates (but does not remove) how many passages are short like the ones described which 
# may disrupt our OCM stats because they contain OCMs without actually having text that refers to these OCMs
shortPass_list = []
for i in df['Passage']:
    if len(i)<=10:
        shortPass_list.append(i)
print(f'Number of passages with text with 10 or fewer characters: {len(shortPass_list)}')

Number of passages with text with 10 or fewer characters: 77


## (OPTIONAL) Reorganize passages by source


The following code reorganizes the passages so that sources are split up rather than clumped together in the same place.<br>
Note that this code wile only take into account the source you want to filter. This means that the same author between different documents will be lumped sequentially together. <br>

<fontcolor=Blue>OTE:</font>

### Create Inermediate dataframe (also allow for optional dataframe for testing)

In [222]:
# Uncomment one of these, not both

# get original dataset
df_sourceSort = df.copy()

# # OPTIONAL get dummy subsample dataset for testing
# addedPass = 0 # CHANGE this number if you want the dummy dubsample to have more rows than just 30
# x = list(range(10,20+addedPass)) + list(range(2050,2060+addedPass))+ list(range(5550,5560+addedPass))
# df_sourceSort = df.iloc[x].copy()
# df_sourceSort = df_sourceSort.sort_values(by='Passage Number')



### Get optional inputs

In [223]:
# CHANGE enter in the column name of the type of source you want to split. Viable options are 'DocTitle', 'Section', 'Author', or really any column name you want
sourceType = 'Author'
# CHANGE enter in the number of passages per source (will loop untill all passages are organized)
n_sources = 3
# CHANGE if you are using the 'Author' source. Remove years. You can combine same named authors but from different years 
# into one source (e.g. 'Dumont, Fred, 1950-1960' and 'Dumont, Fred, 1980-2000' just become 'Dumont, Fred'
rmvAuthorYears_bool = True #change to True or False
# CHANGE Do you want the Cultures to stay in order (setting this to True organize the sources within the culture instead of the whole dataset.
# Setting this to False will interleave sources without care if they come from different cultures. Most people will probably want this to be True)
maintainCultureOrder_bool = True

Optionally remove years at the end of author sources 

In [224]:
if rmvAuthorYears_bool == True:
    if sourceType != 'Author':
        input_q = input('Your source is not the Author, are you sure you want to run this? y/n')
        if input_q.lower() != 'y':
            raise Exception('User canceled run')
        else:
            print('Author reduced')
            
    print("Unique sources BEFORE:", len(set(df_sourceSort['Author'])))
    df_sourceSort['Author']= df_sourceSort['Author'].apply(lambda x: re.sub(r',(?=[^,]*\d)[^,]*$', '', x)) # remove the end year if it has a year
    print("Unique sources AFTER:", len(set(df_sourceSort['Author'])))


Unique sources BEFORE: 7
Unique sources AFTER: 7


In [218]:
df_sourceSort #show the subsample passages

,Passage Number,Region,SubRegion,Culture,DocTitle,Section,Author,Page,Year,OCM,OWC,Passage,run_Info
0,1,Africa,Northern Africa,Libyan Bedouin,Writing women's worlds: Bedouin stories,Losing Men,"Abu-Lughod, Lila",57,1993,"[753, 761, 902]",mt09,"“When Jawwad went in to see him, he shuddered....",User: Eric Chantland
1,2,Africa,Northern Africa,Libyan Bedouin,Writing women's worlds: Bedouin stories,Losing Men,"Abu-Lughod, Lila",59,1993,"[164, 752, 902]",mt09,"He had stepped on a mine. “Watch out, watch ou...",Run Time: 15:52:44
2,3,Africa,Northern Africa,Libyan Bedouin,Writing women's worlds: Bedouin stories,Losing Men,"Abu-Lughod, Lila",60,1993,"[164, 752, 902]",mt09,"“We used to go out and collect copper,” he beg...",Run Date: 08/28/23
3,4,Africa,Northern Africa,Libyan Bedouin,Writing women's worlds: Bedouin stories,Losing Men,"Abu-Lughod, Lila",60,1993,"[164, 752, 902]",mt09,“I was walking along when I found that my shoe...,"Run Input: subjects:(""religious practices"" OR ..."
4,5,Africa,Northern Africa,Libyan Bedouin,Writing women's worlds: Bedouin stories,Losing Men,"Abu-Lughod, Lila",61,1993,"[164, 752, 902]",mt09,“It was hissing and there was smoke and after ...,Filter:\nculture level samples|PSF
...,...,...,...,...,...,...,...,...,...,...,...,...,...
132675,132676,North-America,Plains and Plateau,Pawnee,When stars came down to earth: cosmology of th...,OTHER STARS,"Chamberlain, Von Del",134,1982,"[793, 805, 821]",nq18,"Next we consider the Seven Stars, the Pleiades...",NaN
132678,132679,North-America,Plains and Plateau,Pawnee,When stars came down to earth: cosmology of th...,4 STAGING: THE SKIDI OBSERVATIONAL SYSTEM,"Chamberlain, Von Del",175,1982,"[121, 342, 353, 793]",nq18,The Pleiades offer an interesting illustration...,NaN
132679,132680,North-America,Plains and Plateau,Pawnee,When stars came down to earth: cosmology of th...,4 STAGING: THE SKIDI OBSERVATIONAL SYSTEM,"Chamberlain, Von Del",179,1982,"[121, 342, 793]",nq18,The observatory features of the Pawnee house m...,NaN
132681,132682,North-America,Plains and Plateau,Pawnee,The chief and his council: unity and authority...,The United Stars,"Chamberlain, Von Del",229,1992,"[793, 805, 821]",nq18,"The Skidi associated another group of stars, w...",NaN


In [219]:
# show counts for each source
df_sourceSort.value_counts(sourceType)

Author
Lambrecht, Francis                       2047
Adriani, Nicolaus                        1234
Firth, Raymond                            681
Evans-Pritchard, E. E. (Edward Evan),     443
Bohannan, Paul                            391
                                         ... 
Pasternak, Burton                           1
Chervin, Arthur                             1
Padoch, Christine                           1
Norick, Frank Albert                        1
Abbott, P. H.                               1
Name: count, Length: 689, dtype: int64

In [221]:
def sourceInterleave(df_input:pd.DataFrame, df_output:pd.DataFrame, n_sources:int):
    while len(df_input) >0:
        # Get N rows and get the index of these rows to later drop
        df_sourceSectional = df_input.groupby(sourceType).head(n_sources)
        sourceSectional_index = df_sourceSectional.index
        
        df_output = pd.concat([df_output, df_sourceSectional], ignore_index=True)

        #drop the used indexes
        df_input = df_input.drop(sourceSectional_index)
    return df_output

# n_sources = n_sources
n_sources = 3


df_output = pd.DataFrame([])
if maintainCultureOrder_bool == True: #maintain the culture order (see above 'Get optional inputs' for where this is defined)
    for cult in df_sourceSort['Culture'].unique():
        df_cult = df_sourceSort.loc[df_sourceSort['Culture']==cult].copy()
        df_output = sourceInterleave(df_input=df_cult,df_output=df_output, n_sources=n_sources)
    df_sourceSort = df_output.copy()
else:
    df_sourceSort = sourceInterleave(df_input=df_sourceSort,df_output=df_output, n_sources=n_sources)
df_sourceSort

,Passage Number,Region,SubRegion,Culture,DocTitle,Section,Author,Page,Year,OCM,OWC,Passage,run_Info
0,1,Africa,Northern Africa,Libyan Bedouin,Writing women's worlds: Bedouin stories,Losing Men,"Abu-Lughod, Lila",57,1993,"[753, 761, 902]",mt09,"“When Jawwad went in to see him, he shuddered....",User: Eric Chantland
1,2,Africa,Northern Africa,Libyan Bedouin,Writing women's worlds: Bedouin stories,Losing Men,"Abu-Lughod, Lila",59,1993,"[164, 752, 902]",mt09,"He had stepped on a mine. “Watch out, watch ou...",Run Time: 15:52:44
2,3,Africa,Northern Africa,Libyan Bedouin,Writing women's worlds: Bedouin stories,Losing Men,"Abu-Lughod, Lila",60,1993,"[164, 752, 902]",mt09,"“We used to go out and collect copper,” he beg...",Run Date: 08/28/23
3,36,Africa,Northern Africa,Libyan Bedouin,The Sanusi of Cyrenaica,I,"Evans-Pritchard, Edward Evan",4,1949,"[157, 786, 793, 795]",mt09,As this is not a treatise on Sufism there is n...,NaN
4,38,Africa,Northern Africa,Libyan Bedouin,The Sanusi of Cyrenaica,II,"Evans-Pritchard, Edward Evan",6,1949,"[114, 272, 273, 276, 533, 535, 784, 793, 795]",mt09,"It is true that the Grand Sanusi, like the fou...",NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
22317,132463,North-America,Plains and Plateau,Pawnee,Ceremonies of the Pawnee,Notes on the Songs and Their Composers,"Murie, James R.",467,1989,"[533, 793, 821]",nq18,"As the priest sings on he sings faster, for no...",NaN
22318,132464,North-America,Plains and Plateau,Pawnee,Ceremonies of the Pawnee,Notes on the Songs and Their Composers,"Murie, James R.",467,1989,"[533, 793, 821]",nq18,"There are songs about the moon, the sun, and t...",NaN
22319,132577,North-America,Plains and Plateau,Pawnee,Pawnee Indian societies,THE BUNDLE SCHEME.,"Murie, James R.",555,1914,[793],nq18,The priests of the four leading bundles and th...,NaN
22320,132578,North-America,Plains and Plateau,Pawnee,Pawnee Indian societies,THE BUNDLE SCHEME.,"Murie, James R.",555,1914,"[184, 793]",nq18,"In this connection, it may be noted that the c...",NaN


In [ ]:
# save new sorting to df
input_q = input('you are about to overwrite the original dataframe with a new sorting. Are you sure? y/n')
if input_q.lower() == 'y':
    df = df_sourceSort.copy()
    print('df overwritten')
else:
    print('overwrite CANCELED')

## Save File

In [29]:
df

,Passage Number,Region,SubRegion,Culture,DocTitle,Section,Author,Page,Year,OCM,OWC,Passage
0,1,Africa,Northern Africa,Libyan Bedouin,Writing women's worlds: Bedouin stories,Losing Men,"Abu-Lughod, Lila",57,1993,"[753, 761, 902]",mt09,"“When Jawwad went in to see him, he shuddered...."
1,2,Africa,Northern Africa,Libyan Bedouin,Writing women's worlds: Bedouin stories,Losing Men,"Abu-Lughod, Lila",59,1993,"[164, 752, 902]",mt09,"He had stepped on a mine. “Watch out, watch ou..."
2,3,Africa,Northern Africa,Libyan Bedouin,Writing women's worlds: Bedouin stories,Losing Men,"Abu-Lughod, Lila",60,1993,"[164, 752, 902]",mt09,"“We used to go out and collect copper,” he beg..."
3,4,Africa,Northern Africa,Libyan Bedouin,Writing women's worlds: Bedouin stories,Losing Men,"Abu-Lughod, Lila",60,1993,"[164, 752, 902]",mt09,“I was walking along when I found that my shoe...
4,5,Africa,Northern Africa,Libyan Bedouin,Writing women's worlds: Bedouin stories,Losing Men,"Abu-Lughod, Lila",61,1993,"[164, 752, 902]",mt09,“It was hissing and there was smoke and after ...
...,...,...,...,...,...,...,...,...,...,...,...,...
132675,132676,North-America,Plains and Plateau,Pawnee,When stars came down to earth: cosmology of th...,OTHER STARS,"Chamberlain, Von Del",134,1982,"[793, 805, 821]",nq18,"Next we consider the Seven Stars, the Pleiades..."
132678,132679,North-America,Plains and Plateau,Pawnee,When stars came down to earth: cosmology of th...,4 STAGING: THE SKIDI OBSERVATIONAL SYSTEM,"Chamberlain, Von Del",175,1982,"[121, 342, 353, 793]",nq18,The Pleiades offer an interesting illustration...
132679,132680,North-America,Plains and Plateau,Pawnee,When stars came down to earth: cosmology of th...,4 STAGING: THE SKIDI OBSERVATIONAL SYSTEM,"Chamberlain, Von Del",179,1982,"[121, 342, 793]",nq18,The observatory features of the Pawnee house m...
132681,132682,North-America,Plains and Plateau,Pawnee,The chief and his council: unity and authority...,The United Stars,"Chamberlain, Von Del",229,1992,"[793, 805, 821]",nq18,"The Skidi associated another group of stars, w..."


In [ ]:
print(f'Passages after all cleaning:\n{len(df)}')
# Save the cleaned version of the dataframe TO THE ORIGINAL DIRECTORY
df.to_excel(directory + "/_Altogether_Dataset_CLEANED.xlsx", index=False)

# save the read me file to indicate if the cleaned file was composed of multiple datasets
# with open(directory+'/_README.txt', 'w') as f:
#     f.write(readme_text)

Passages after all cleaning:
109022


## Differentiating Datasets


This is so if you have multiple datasets you want to keep track of, you can print out an excel sheet which will save which passage matches with a dataset<br>
Note that the first dataset in the heirarchy will always take precedence over lower datasets meaning that a passage which could be contained in the first and second dataset will solely be placed in the first!

In [141]:
def dataset_tracker(df, dataset_dict:dict):
    df_datasets = df[["Passage Number", "OCM"]]
    df_datasets.loc[:, ["Dataset","DatasetSplitInfo"]] = ''

    for key in reversed(list(dataset_dict.keys())):
        if len(dataset_dict[key]) == 0:
            continue
        
        OCM_validity_checker(dataset_dict[key])
        
        msk = df['OCM'].apply(lambda x: not set(x).isdisjoint(dataset_dict[key]))
        df_datasets.loc[msk, "Dataset"] = key

    print(f"Total: {len(df_datasets)}")
    df_datasets.loc[0, "DatasetSplitInfo"] = f"Total Count: {len(df_datasets)}"
    counter = len(df_datasets)
    for index, key in enumerate(dataset_dict.keys()):
        if len(dataset_dict[key]) == 0:
            continue
        data_count = len(df_datasets.loc[df_datasets['Dataset']==key])
        counter -= data_count
        print(f"Dataset {key}: {data_count}")
        df_datasets.loc[index+1, "DatasetSplitInfo"] = f"Dataset {key}: {dataset_dict[key]}   Count: {data_count}"

    
    if counter != 0:
        print("\n\n\033[91m{}\033[00m".format(f"WARNING number of passages do not add up to total:\n{counter}"))
        print("Make sure this is okay as it likely means your OCM codes you put in do not match the filtering done in the cleaning step")
    # If the passage numbers have been updated, add the column to the dataset as these number may be important to use later
    if redonePassageNums_bool:
        df_datasets = df_datasets.copy() #Here to suppress the slice warning but I am note sure why this fixes it.
        passLoc = df_datasets.columns.get_loc("Passage Number")
        df_datasets.insert(passLoc+1,'Passage Number Original', origPassageNums.loc[df.index])
    return df_datasets
# df['OCM'].apply(lambda x: not set(x).isdisjoint(["780", "781", "784", "785", "788"]))
    

In [142]:

# CHANGE insert any number of OCMs per dictionary list. You may add more datasets if you want (keep the same format shown here) or even chnage the name of the dictionary (does not have to be 1,2,3,4)
# The higher in order the list is, the more it will take precedence when deciding which dataset a passage is located in (when a passage could be contained in more than one dataset)

# # Datasets include misfortune datasets, 780 dataset, theoretical interest dataset, and non theoretical interest
# dataset_dict = {"1":["750", "751", "752", "753"], 
#                 "2":["780", "781", "784", "785"],
#                 "3":['586' , '684' , '688' , '731' , '732' , '756' , '767' , '777' , '791' , '792' , '793'],
#                 "4":['431' , '572' , '594' , '613' , '624' , '675' , '853']}

# dataset marking misfortune dataset and 
dataset_dict = {"1":["750", "751", "752", "753"], 
                "2":['784' , '731' , '732' , '777' , '791', '793'],
                "3":[],
                "4":[]}


  
df_datasets = dataset_tracker(df, dataset_dict)


# double check to make sure you are not overwriting the file you actually do not want to
print('\n')
if os.path.exists(directory+"/_Dataset_Lists.xlsx"):
    double_check = input("Are you sure you want to overwite the current dataset list? y/n")
    if double_check.lower() == 'y':
        df_datasets.to_excel(directory+"/_Dataset_Lists.xlsx", index=False)
        print('Dataset Overwritten')
    else:
        print('New dataset not saved')
else:
    df_datasets.to_excel(directory+"/_Dataset_Lists.xlsx", index=False)
    print('New dataset saved!')

Total: 22322
Dataset 1: 6112
Dataset 2: 16210


New dataset not saved


## (Optional) OCM Code Counting

Count every OCM within each culture. Do not count OCM's specified by the search (like if searched for 750-755, do not count these). 
<!-- - REMOVE all passages which are blank since we can't very well do lexical searches on them -->

In [44]:
# Make a copy of df_OCM as to not interfere with other analysis
df_OCM_freq = df_OCM.copy()
# Then turn the OCM's back to an integer (for removals)
df_OCM_freq['OCM'] = df_OCM_freq.OCM.apply(lambda x: int(x))
# only keep OCMs outside our search parameters whatever those are (make sure OCM_list has been ran above)
df_sub_ex = df_OCM_freq.copy()
for OCM in OCM_list:
    df_sub_ex = df_sub_ex.loc[df_sub_ex["OCM"] != OCM]

# Overwrite and create a new dataframe for OCM counts and frequencies
df_OCM_freq = pd.DataFrame(columns=["Culture","OCM","Frequency","Proportion_of_Passages"])
for key, val in culture_dict.items():
    value_count = df_sub_ex.loc[df_sub_ex["Culture"]==key]["OCM"].value_counts()
    # duplicate the culture word and asign it to each of its rows
    cult_count = [key] * len(value_count)
    # create a culture dataframe and append it to to the 
    df_OCM_Concat = pd.DataFrame({"Culture":cult_count,"OCM":value_count.index, "Frequency":value_count.values, "Proportion_of_Passages":value_count.values/val})
    df_OCM_freq = pd.concat([df_OCM_freq, df_OCM_Concat], ignore_index=True)
df_OCM_freq = df_OCM_freq.sort_values(by = ["Culture", "Frequency"], ascending= [True, False])
df_OCM_freq

,Culture,OCM,Frequency,Proportion_of_Passages
3239,Akan,784,396,0.436604
3240,Akan,793,340,0.374862
3241,Akan,159,230,0.253583
3242,Akan,787,162,0.178611
3243,Akan,778,160,0.176406
...,...,...,...,...
1892,Yanoama,153,1,0.008850
1893,Yanoama,191,1,0.008850
1894,Yanoama,734,1,0.008850
1895,Yanoama,888,1,0.008850


In [45]:
print(f'OCMs per culture: {sum(df_OCM_freq["Frequency"]) / len(set(df_OCM_freq["Culture"]))}')

OCMs per culture: 1461.5333333333333


In [46]:
# Save the file
df_OCM_freq.to_excel(directory+'/'+ "_Culture_Frequency.xlsx", index=False)

## (Optional) Association Rules for OCMs

In [47]:
# Load resources
from mlxtend.preprocessing import TransactionEncoder

# We will use the apriori module to generate a dataframe that
# we can use for association rule finding
from mlxtend.frequent_patterns import apriori

# We will use the association_rules module to generate
# our association rules from the apriori output data frame
from mlxtend.frequent_patterns import association_rules





In [58]:
#Display important columns
df_smaller = df_OCM[['Culture', 'OCM','Passage']]
df_smaller

,Culture,OCM,Passage
0,Libyan Bedouin,753,"“When Jawwad went in to see him, he shuddered...."
1,Libyan Bedouin,761,"“When Jawwad went in to see him, he shuddered...."
2,Libyan Bedouin,902,"“When Jawwad went in to see him, he shuddered...."
3,Libyan Bedouin,164,"He had stepped on a mine. “Watch out, watch ou..."
4,Libyan Bedouin,752,"He had stepped on a mine. “Watch out, watch ou..."
...,...,...,...
242164,Pawnee,805,"The Skidi associated another group of stars, w..."
242165,Pawnee,821,"The Skidi associated another group of stars, w..."
242166,Pawnee,793,When all the invited guests had taken their pl...
242167,Pawnee,796,When all the invited guests had taken their pl...


In [59]:
# created a grouped dataframe object by Culture and Passage 
df_group = df_smaller.groupby(by = ['Culture', 'Passage'])
df_group

In [60]:
def make_OCM_list(x):

    '''
    Will return a list of the unique items
    in a particular grouping when used with
    the agg method as its function
    '''

    return x.unique()

In [61]:
# Use the agg method and make_OCM_list
# to return a list of unique items for each ocm
# Note that depending on the filtering, there may be duplicate passages with different OCMs which are aggregated, 
# this method will combine them and extract the unique OCMs so it may not be a problem.
df_unique = df_group.agg(make_OCM_list)

In [62]:
list_trans = list(df_unique['OCM'])
list_trans = list_trans[0:]
len(list_trans)

66259

In [63]:
te = TransactionEncoder()
encoded_itemset = te.fit(list_trans).transform(list_trans)
print(encoded_itemset.shape) # show possible transcations and number of items
te.columns_



df_encoded = pd.DataFrame(encoded_itemset, columns = te.columns_)
df_encoded.head()

(66259, 674)


,0,101,102,103,104,105,106,111,112,113,...,901,902,903,905,924,931,947,978,981,984
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [64]:
# Before we begin, let's do a small
# amount of cleanup.  Let's remove all
# columns (items) that have less than 1 characters since that is just blank space
# more data cleaning my be required as time continues in case errors become evident in the scraped dataset
OCM_items = list(filter(lambda x: len(x) < 1, te.columns_ ))
print("removed: ",  OCM_items)
df_encoded = df_encoded.drop(columns=OCM_items) #remove small strings as they seem not to be items
print('How many unique items are left?', len(df_encoded.columns))

removed:  []
How many unique items are left? 674


In [65]:
# Use apriori to create a dataframe with columns of support and itemset lists
# Note that if your items are large compared to your sample (you have few rows but many columns) I reccommend using 
# a higher min_support as many more combinations may have spuriously higher support. Also, you can crash the program if too many are selected
df_support = apriori(df_encoded, min_support=0.01, use_colnames=True)
df_support.sort_values('support', inplace=True, ascending = False)
df_support

,support,itemsets
51,0.205995,(756)
36,0.154062,(613)
15,0.120753,(431)
73,0.111970,(796)
71,0.110400,(793)
...,...,...
28,0.010338,(582)
62,0.010187,(779)
14,0.010157,(428)
30,0.010097,(585)


### Use association_rules to find the rules

Using the dataframe generated by `apriori`, find the association rules with the greatest lift.  See the [association_rules documentation](https://rasbt.github.io/mlxtend/api_modules/mlxtend.frequent_patterns/association_rules/) for how to do this.

Sort the resulting DataFrame by lift in descending order.  A lift > 1 indicates that the items are often purchased together and that buying X will increase the purchase of Y.  A lift of < 1 indicates the items are often substituted.  That is X is substituted for Y so X and Y don't appear together often.

Examine the resulting DataFrame.  For the association rule X -> Y, X is the column `antecedents` and Y is the column `consequents`.  If sorted you can see the metrics for each rule based upon the lift.

In [77]:
# Find the association rules
rules = association_rules(df_support, metric = 'lift', min_threshold=1)
# lift >1 more likely than chance X means you see Y
# lift = 1 as often as chance
# lift <1 (substitution) less likely than chance X means you see Y


In [78]:
# Sort the rules by lift
# and examine the output
# to find what rules were
# discovered
rules.sort_values('lift', ascending=False, inplace =True)
rules

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,leverage,conviction,zhangs_metric
623,"(776, 539, 782)","(775, 755, 793)",0.018473,0.019031,0.014307,0.774510,40.696467,0.013956,4.350383,0.993786
642,"(775, 755, 793)","(776, 539, 782)",0.019031,0.018473,0.014307,0.751784,40.696467,0.013956,3.954331,0.994352
640,"(775, 539, 755)","(776, 782, 793)",0.015681,0.022669,0.014307,0.912416,40.250171,0.013952,11.158762,0.990690
625,"(776, 782, 793)","(775, 539, 755)",0.022669,0.015681,0.014307,0.631158,40.250171,0.013952,2.668677,0.997774
637,"(775, 782, 755)","(776, 539, 793)",0.018986,0.019258,0.014307,0.753577,39.131086,0.013942,3.979915,0.993304
...,...,...,...,...,...,...,...,...,...,...
670,(796),(782),0.111970,0.068519,0.012134,0.108370,1.581611,0.004462,1.044695,0.414100
402,(796),(793),0.111970,0.110400,0.016466,0.147055,1.332017,0.004104,1.042974,0.280688
403,(793),(796),0.110400,0.111970,0.016466,0.149146,1.332017,0.004104,1.043692,0.280192
140,(776),(756),0.080970,0.205995,0.019560,0.241566,1.172679,0.002880,1.046901,0.160226


In [79]:
# look for OCM codes within the list
lst = frozenset(["793","226"])
msk = rules['antecedents'].apply(lambda x: not set(x).isdisjoint(lst))
out = rules.loc[msk]
out

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,leverage,conviction,zhangs_metric
642,"(775, 755, 793)","(776, 539, 782)",0.019031,0.018473,0.014307,0.751784,40.696467,0.013956,3.954331,0.994352
625,"(776, 782, 793)","(775, 539, 755)",0.022669,0.015681,0.014307,0.631158,40.250171,0.013952,2.668677,0.997774
628,"(776, 539, 793)","(775, 782, 755)",0.019258,0.018986,0.014307,0.742947,39.131086,0.013942,3.816383,0.993579
641,"(775, 539, 793)","(776, 782, 755)",0.019197,0.019243,0.014307,0.745283,38.730751,0.013938,3.850381,0.993249
638,"(775, 782, 793)","(776, 539, 755)",0.022533,0.016496,0.014307,0.634963,38.492245,0.013936,2.694260,0.996474
...,...,...,...,...,...,...,...,...,...,...
55,(793),(776),0.110400,0.080970,0.029973,0.271497,3.353050,0.021034,1.261532,0.788854
683,(793),(778),0.110400,0.037157,0.010504,0.095147,2.560659,0.006402,1.064087,0.685112
59,(793),(539),0.110400,0.109329,0.028072,0.254272,2.325761,0.016002,1.194365,0.640775
679,(793),(787),0.110400,0.058800,0.011304,0.102392,1.741379,0.004813,1.048566,0.478578
